# Redes Neuronales Básicas

### Taller — Programación de redes neuronales desde cero con Python y NumPy

---

**Universidad de Cundinamarca**

**CADI:** Deep Learning — Conceptos

**Estudiante:** Ruby Dayana Cárdenas Gómez

**Docente:** Nidia Stella García Roa

**Repositorio GitHub:** `https://github.com/RubyDayana/redes-neuronales-basicas.git`

---

### Contenido

En este taller se programan tres modelos de redes neuronales desde cero, sin usar ninguna librería especializada de Deep Learning:

1. **El perceptrón** — la neurona artificial más sencilla.
2. **Una red neuronal de una capa** — varias neuronas trabajando en paralelo.
3. **Una red neuronal multicapa** — con capa oculta y aprendizaje por retropropagación.

Por caada modelo se prueba con datos previamente definidos y se explica qué está haciendo internamente.

---
## 0. Glosario de siglas y términos

Antes de empezar, aquí están las siglas y palabras que se van a repetir a lo largo del reporte.

| Sigla o símbolo | Significado | En palabras sencillas |
|---|---|---|
| **IA** | Inteligencia Artificial | Programas que hacen tareas que asociamos con la inteligencia humana |
| **ML** | Machine Learning (aprendizaje automático) | Que el computador aprenda a partir de ejemplos, en vez de seguir reglas escritas una por una |
| **DL** | Deep Learning (aprendizaje profundo) | Una rama del ML que usa redes neuronales con varias capas |
| **RNA** | Red Neuronal Artificial | Un conjunto de neuronas artificiales conectadas entre sí |
| **MLP** | Multi-Layer Perceptron (perceptrón multicapa) | Una red con al menos una capa oculta entre la entrada y la salida |
| **X** | Matriz de entradas | Los datos que se le dan a la red. Una fila por ejemplo, una columna por característica |
| **Y** | Vector de salidas esperadas | La respuesta correcta de cada ejemplo |
| **W** | Matriz de pesos (*weights*) | Qué tan importante es cada entrada para cada neurona |
| **b** | Sesgo (*bias*) | Un valor que se suma aparte y le da flexibilidad a la neurona |
| **z** | Suma ponderada | El resultado de multiplicar entradas por pesos y sumar el sesgo |
| **a** | Activación | El valor que sale de la neurona después de aplicar la función de activación |
| **lr** | Learning rate (tasa de aprendizaje) | Qué tan grandes son los pasos que da la red al corregirse |
| **Época** | *Epoch* | Una pasada completa por todos los datos de entrenamiento |
| **MSE** | Mean Squared Error (error cuadrático medio) | La medida de qué tan equivocada está la red |

______________________________________________


| Término | Qué significa |
|---|---|
| **Neurona artificial** | Una operación que toma varios números, los pondera, los suma y pasa el resultado por una función |
| **Función de activación** | La que decide qué valor entrega la neurona (escalón, sigmoide, etc.) |
| **Capa** | Un grupo de neuronas que trabajan en paralelo sobre las mismas entradas |
| **Capa oculta** | Una capa intermedia, entre la entrada y la salida |
| **Propagación hacia adelante** (*forward*) | Pasar los datos de la entrada hasta la salida para obtener una predicción |
| **Retropropagación** (*backpropagation*) | Devolverse desde el error hasta los pesos para saber cuánto corregir cada uno |
| **Vectorización** | Operar con matrices completas de una sola vez, en lugar de recorrerlas dato por dato |

---
## 1. Conceptos: de Machine Learning a Deep Learning

**Machine Learning (ML)** es la idea de que el computador aprenda a partir de ejemplos. En lugar de que
alguien escriba todas las reglas a mano, se le muestran muchos casos con su respuesta correcta y el
programa deduce solo el patrón.

**Deep Learning (DL)** es una rama del ML que usa redes neuronales con varias capas. La palabra
"profundo" se refiere justamente a eso: a la cantidad de capas apiladas una detrás de otra. Cada capa va
detectando algo un poco más elaborado que la anterior.

### ¿Qué es una neurona artificial?

Es mucho más simple de lo que suena. Una neurona recibe varios números de entrada, y a cada uno le asigna
un peso, que indica qué tan importante es. Multiplica cada entrada por su peso, suma todo, le agrega un
valor extra llamado sesgo, y pasa ese resultado por una función de activación que produce la salida final:

$$z = x_1w_1 + x_2w_2 + \dots + x_nw_n + b \qquad\qquad a = f(z)$$

Aprender, para una red neuronal, significa ir ajustando esos pesos hasta que las predicciones se
parezcan a las respuestas correctas. Nada más.

In [ ]:
# Única librería que vamos a usar en este taller
import numpy as np
import time

# Fijamos la semilla para que los resultados no cambien cada vez que se ejecuta
np.random.seed(42)

print("NumPy version:", np.__version__)

NumPy version: 2.1.3


---
## 2. Operaciones con NumPy y vectorización

Antes de programar la red hay que entender la herramienta. Todo lo que hace una red neuronal son multiplicaciones y sumas de matrices, y eso es exactamente lo que NumPy hace bien.

**Vectorizar** significa operar con la tabla completa de una sola vez, en lugar de recorrerla fila por
fila con un ciclo. No es solo cuestión de escribir menos código: es muchísimo más rápido.

### 2.1. Vectores, matrices y la forma de los datos

En una red neuronal, la **forma** (*shape*) de cada matriz es lo que más se revisa, porque casi todos los
errores vienen de intentar multiplicar matrices que no encajan.

In [ ]:
# Un vector: las dos entradas de un solo ejemplo
x = np.array([0.5, 0.8])

# 1) MATRICES: cada fila es un ejemplo, cada columna una caracteristica
# Una matriz: cuatro ejemplos, cada uno con dos entradas (4 filas x 2 columnas)
X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]])

print("Vector x:", x, "-> forma:", x.shape)
print()
print("Matriz X:")
print(X)
print("-> forma:", X.shape, " (4 ejemplos, 2 caracteristicas cada uno)")

Vector x: [0.5 0.8] -> forma: (2,)

Matriz X:
[[0 0]
 [0 1]
 [1 0]
 [1 1]]
-> forma: (4, 2)  (4 ejemplos, 2 caracteristicas cada uno)


### 2.2. El producto punto: el corazón de la neurona

La operación `x · w` (multiplicar cada entrada por su peso y sumar todo) se llama **producto punto**.
Para comprobar que NumPy hace exactamente lo que dice la fórmula, vamos a calcularlo de las dos formas:
a mano con un ciclo, y con NumPy.

In [ ]:
# 2) PRODUCTO PUNTO: multiplicar cada entrada por su peso y sumar
entradas = np.array([1.0, 2.0, 3.0])
pesos    = np.array([0.4, -0.2, 0.7])

# Forma 1: a mano, recorriendo elemento por elemento
suma_manual = 0
for i in range(len(entradas)):
    suma_manual += entradas[i] * pesos[i]

# Forma 2: vectorizada, con NumPy
suma_numpy = np.dot(entradas, pesos)

print("Calculo manual :", suma_manual)
print("Con np.dot     :", suma_numpy)
print("Son iguales?   :", np.isclose(suma_manual, suma_numpy))

Calculo manual : 2.0999999999999996
Con np.dot     : 2.0999999999999996
Son iguales?   : True


### 2.3. Multiplicación de matrices: todos los ejemplos a la vez

Aquí está la verdadera ventaja. Si `X` tiene 4 ejemplos y `W` tiene los pesos de 3 neuronas, la operación
`X @ W` calcula las 12 respuestas de una sola vez (4 ejemplos × 3 neuronas), sin escribir un solo ciclo.

La regla para que dos matrices se puedan multiplicar: las columnas de la primera deben ser iguales a las
filas de la segunda.

In [ ]:
# 3) TODAS LAS NEURONAS A LA VEZ: X @ W calcula todos los ejemplos y todas las neuronas de una sola vez
X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]]) # 4 ejemplos x 2 entradas

W = np.array([[0.5, -0.3, 0.8],
              [0.2,  0.9, -0.4]]) # 2 entradas x 3 neuronas

b = np.array([0.1, 0.0, -0.2]) # un sesgo por neurona

Z = X @ W + b # el operador @ multiplica matrices; el sesgo se suma por ("Broadcasting")solo a cada fila

print("Forma de X:", X.shape, " x  forma de W:", W.shape, " =  forma de Z:", Z.shape)
print()
print("Resultado Z (4 ejemplos x 3 neuronas):")
print(np.round(Z, 3))

Forma de X: (4, 2)  x  forma de W: (2, 3)  =  forma de Z: (4, 3)

Resultado Z (4 ejemplos x 3 neuronas):
[[ 0.1  0.  -0.2]
 [ 0.3  0.9 -0.6]
 [ 0.6 -0.3  0.6]
 [ 0.8  0.6  0.2]]


### 2.4. ¿Qué es Broadcasting?

Supongamos que `b` tiene 3 valores y `Z` tiene 4 filas de 3 valores. NumPy suma automáticamente el sesgo a
cada una de las filas, sin que haya que repetirlo cuatro veces. A eso se le llama *broadcasting*.

In [ ]:
matriz = np.array([[1, 2, 3],
                   [4, 5, 6]])
sesgo  = np.array([10, 20, 30])

print("Matriz original:")
print(matriz)
print()
print("Matriz + sesgo (el sesgo se suma a cada fila automaticamente):")
print(matriz + sesgo)

Matriz original:
[[1 2 3]
 [4 5 6]]

Matriz + sesgo (el sesgo se suma a cada fila automaticamente):
[[11 22 33]
 [14 25 36]]


---
## 3. El perceptrón

El **perceptrón** es la neurona artificial más antigua y más sencilla. Funciona de la siguiente forma:

1. Recibe las entradas y las multiplica por sus pesos.
2. Suma todo junto con el sesgo.
3. Si el resultado es mayor o igual a cero, responde **1**; si no, responde **0**.

Ese último paso es la función escalón: no hay término medio, la respuesta es sí o no.

### ¿Cómo aprende?

Con una regla muy intuitiva: se le muestra un ejemplo, se compara su respuesta con la correcta y, si se
equivocó, se corrigen los pesos en la dirección del error:

$$w_{nuevo} = w + \text{lr} \times (\text{esperado} - \text{obtenido}) \times x$$

Si acertó, el error es cero y los pesos no cambian. Si se equivocó, se mueven un poquito. Se repite el
proceso muchas veces hasta que no queden errores.

In [ ]:
def escalon(z):
    """Funcion de activacion escalon: devuelve 1 si z >= 0, y 0 en caso contrario.

    np.where recorre todo el arreglo de una vez (vectorizacion), sin necesidad de un ciclo.
    """
    return np.where(z >= 0, 1, 0)


def predecir_perceptron(X, W, b):
    """Calcula la salida del perceptron para todos los ejemplos a la vez.

    X @ W  -> producto matricial: multiplica cada entrada por su peso y suma
    + b    -> agrega el sesgo a cada ejemplo (broadcasting)
    """
    z = X @ W + b
    return escalon(z)


# Prueba rapida con pesos inventados, solo para ver que funciona
X_prueba = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
W_prueba = np.array([1.0, 1.0])
b_prueba = -1.5

print("Entradas  ->  Salida")
for entrada, salida in zip(X_prueba, predecir_perceptron(X_prueba, W_prueba, b_prueba)):
    print(f"  {entrada}   ->    {salida}")

Entradas  ->  Salida
  [0 0]   ->    0
  [0 1]   ->    0
  [1 0]   ->    0
  [1 1]   ->    1


### 3.1. La función de entrenamiento

Ahora programamos el aprendizaje. La función recorre los datos varias veces (épocas) y va corrigiendo los
pesos cada vez que el perceptrón se equivoca.

In [ ]:
def entrenar_perceptron(X, Y, lr=0.1, epocas=20, mostrar=True):
    """Entrena un perceptron con la regla de aprendizaje clasica.

    X  : matriz de entradas  (n_ejemplos x n_entradas)
    Y  : vector de salidas esperadas (n_ejemplos)
    lr : tasa de aprendizaje, el tamano del paso de correccion
    """
    n_entradas = X.shape[1]

    # Pesos iniciales pequenos y aleatorios; el sesgo arranca en cero
    W = np.random.uniform(-0.5, 0.5, size=n_entradas)
    b = 0.0

    historial = [] # guardamos cuantos errores hubo en cada epoca

    for epoca in range(epocas):
        errores = 0
        for entrada, esperado in zip(X, Y):
            obtenido = escalon(np.dot(entrada, W) + b) # prediccion de este ejemplo
            error = esperado - obtenido # 0 si acerto, +1 o -1 si fallo

            # Correccion: solo cambia algo si hubo error
            W = W + lr * error * entrada
            b = b + lr * error

            errores += int(error != 0)

        historial.append(errores)
        if mostrar:
            print(f"  Epoca {epoca + 1:2d} -> errores: {errores}")

        if errores == 0: # ya no se equivoca en ningun ejemplo: terminamos
            if mostrar:
                print(f"  Aprendio en {epoca + 1} epocas.")
            break

    return W, b, historial


def tabla_resultados(X, Y, W, b, titulo):
    """Imprime una tabla comparando lo esperado con lo que responde el perceptron."""
    predicciones = predecir_perceptron(X, W, b)
    print(f"\n{titulo}")
    print("  Entrada     Esperado   Obtenido   ?")
    for entrada, esperado, obtenido in zip(X, Y, predicciones):
        marca = "OK" if esperado == obtenido else "FALLA"
        print(f"  {entrada}        {esperado}          {obtenido}      {marca}")
    aciertos = np.sum(predicciones == Y)
    print(f"  Aciertos: {aciertos} de {len(Y)}")
    print(f"  Pesos aprendidos: W = {np.round(W, 3)}, b = {round(float(b), 3)}")

### 3.2. Prueba 1: Compuerta AND y Compuerta OR

La compuerta lógica **AND** solo responde 1 cuando las dos entradas son 1. Es el ejemplo clásico porque
los datos son mínimos y se puede verificar el resultado a simple vista.

La compuerta **OR** responde 1 cuando al menos una de las entradas es 1.

In [ ]:
# Datos predefinidos: las cuatro combinaciones posibles de dos entradas
X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]])

Y_and = np.array([0, 0, 0, 1]) # solo 1 cuando ambas entradas son 1
print("Entrenando el perceptron con la compuerta AND:")
W_and, b_and, hist_and = entrenar_perceptron(X, Y_and, lr=0.1, epocas=20)

tabla_resultados(X, Y_and, W_and, b_and, "RESULTADO - Compuerta AND")

Y_or = np.array([0, 1, 1, 1])
print("____________________________________________________")
print("\nEntrenando el perceptron con la compuerta OR:")
W_or, b_or, hist_or = entrenar_perceptron(X, Y_or, lr=0.1, epocas=20)

tabla_resultados(X, Y_or, W_or, b_or, "\nRESULTADO - Compuerta OR")

Entrenando el perceptron con la compuerta AND:
  Epoca  1 -> errores: 2
  Epoca  2 -> errores: 2
  Epoca  3 -> errores: 2
  Epoca  4 -> errores: 1
  Epoca  5 -> errores: 0
  Aprendio en 5 epocas.

RESULTADO - Compuerta AND
  Entrada     Esperado   Obtenido   ?
  [0 0]        0          0      OK
  [0 1]        0          0      OK
  [1 0]        0          0      OK
  [1 1]        1          1      OK
  Aciertos: 4 de 4
  Pesos aprendidos: W = [0.075 0.251], b = -0.3
____________________________________________________

Entrenando el perceptron con la compuerta OR:
  Epoca  1 -> errores: 2
  Epoca  2 -> errores: 1
  Epoca  3 -> errores: 0
  Aprendio en 3 epocas.


RESULTADO - Compuerta OR
  Entrada     Esperado   Obtenido   ?
  [0 0]        0          0      OK
  [0 1]        1          1      OK
  [1 0]        1          1      OK
  [1 1]        1          1      OK
  Aciertos: 4 de 4
  Pesos aprendidos: W = [0.232 0.199], b = -0.1


### 3.3. Prueba 2: Compuerta XOR

La compuerta **XOR** responde 1 solo cuando las entradas son distintas. Parece igual de sencilla que
las anteriores… pero el perceptrón no puede resolverla, por más épocas que se le den.

La razón es geométrica: un perceptrón traza una sola línea recta para separar los casos que responden
0 de los que responden 1. Con AND y OR eso se puede hacer. Con XOR no existe ninguna línea recta que deje
de un lado los puntos (0,1) y (1,0) y del otro los puntos (0,0) y (1,1).

Este fue un problema histórico real que frenó la investigación en redes neuronales durante años, y la
solución fue justamente agregar capas, que es lo que veremos más adelante.

In [ ]:
Y_xor = np.array([0, 1, 1, 0]) # responde 1 solo si las entradas son diferentes

print("Intentando entrenar el perceptron con XOR (30 epocas):")
W_xor, b_xor, hist_xor = entrenar_perceptron(X, Y_xor, lr=0.1, epocas=30, mostrar=False)

print(f"  Errores en la primera epoca : {hist_xor[0]}")
print(f"  Errores en la ultima epoca  : {hist_xor[-1]}")
print(f"  Epocas ejecutadas           : {len(hist_xor)}  (nunca llego a 0 errores)")

tabla_resultados(X, Y_xor, W_xor, b_xor, "RESULTADO - Compuerta XOR (no lo logra)")

Intentando entrenar el perceptron con XOR (30 epocas):
  Errores en la primera epoca : 3
  Errores en la ultima epoca  : 4
  Epocas ejecutadas           : 30  (nunca llego a 0 errores)

RESULTADO - Compuerta XOR (no lo logra)
  Entrada     Esperado   Obtenido   ?
  [0 0]        0          1      FALLA
  [0 1]        1          0      FALLA
  [1 0]        1          0      FALLA
  [1 1]        0          0      OK
  Aciertos: 1 de 4
  Pesos aprendidos: W = [-0.244 -0.144], b = 0.1


---
## 4. Red neuronal de una capa

Un perceptrón es una sola neurona. Una capa es un grupo de neuronas que reciben las mismas
entradas pero tienen pesos distintos, así que cada una aprende algo diferente y todas responden al tiempo.

Aquí cambian dos cosas respecto al perceptrón:

1. Los pesos ya no son un vector sino una matriz: `W` tiene una columna por cada neurona.
2. Cambiamos la función escalón por la sigmoide, que en vez de responder solo 0 o 1 entrega cualquier
   valor entre 0 y 1 — algo así como "estoy 80 % seguro". Eso permite medir qué tan equivocada está la red
   y corregirla de forma más fina.

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

In [ ]:
def sigmoide(z):
    """Convierte cualquier numero en un valor entre 0 y 1."""
    return 1 / (1 + np.exp(-z))


def derivada_sigmoide(a):
    """Derivada de la sigmoide,
       Se usa para saber en que direccion mover los pesos al corregir el error.
    """
    return a * (1 - a)


# Ejemplo de como funcionaa
valores = np.array([-6, -2, -0.5, 0, 0.5, 2, 6])
print(" z      sigmoide(z)")
for z, s in zip(valores, sigmoide(valores)):
    print(f"{z:5.1f}      {s:.4f}")
print()
print("Observa: valores muy negativos se acercan a 0, muy positivos a 1, y en z=0 da exactamente 0.5")

 z      sigmoide(z)
 -6.0      0.0025
 -2.0      0.1192
 -0.5      0.3775
  0.0      0.5000
  0.5      0.6225
  2.0      0.8808
  6.0      0.9975

Observa: valores muy negativos se acercan a 0, muy positivos a 1, y en z=0 da exactamente 0.5


### 4.1. Entrenando una capa completa

Vamos a ahcer algo que el perceptrón no podía: entrenar dos neuronas al mismo tiempo, una que aprenda
AND y otra que aprenda OR, con una sola matriz de pesos.

El ajuste ya no se hace ejemplo por ejemplo, sino con todos los datos a la vez usando operaciones
matriciales. Este es el aprendizaje por descenso del gradiente: se calcula el error, se mira en qué
dirección habría que mover cada peso para reducirlo, y se da un pasito en esa dirección.

In [ ]:
def entrenar_capa(X, Y, lr=0.5, epocas=5000):
    """Entrena una capa de neuronas de forma totalmente vectorizada.

    X : (n_ejemplos x n_entradas)
    Y : (n_ejemplos x n_neuronas)  -> una columna por cada cosa que queremos predecir
    """
    n_entradas = X.shape[1]
    n_neuronas = Y.shape[1]

    # Matriz de pesos: una fila por entrada, una columna por neurona
    W = np.random.uniform(-1, 1, size=(n_entradas, n_neuronas))
    b = np.zeros((1, n_neuronas))

    for epoca in range(epocas):
        # --- Propagacion hacia adelante: de la entrada a la prediccion ---
        Z = X @ W + b  # suma ponderada de TODOS los ejemplos de una vez
        A = sigmoide(Z) # prediccion de cada neurona

        # --- Medimos el error ---
        error = A - Y # cuanto se paso o se quedo corta cada neurona
        costo = np.mean(error ** 2) # error cuadratico medio (MSE)

        # --- Correccion de los pesos (descenso del gradiente) ---
        delta = error * derivada_sigmoide(A)  # cuanta "culpa" tiene cada neurona
        gradiente_W = X.T @ delta / X.shape[0] # cuanto debe cambiar cada peso
        gradiente_b = np.mean(delta, axis=0, keepdims=True)

        W = W - lr * gradiente_W # se resta porque queremos DISMINUIR el error
        b = b - lr * gradiente_b

        if epoca % 1000 == 0:
            print(f"  Epoca {epoca:5d} -> error (MSE): {costo:.6f}")

    print(f"  Epoca {epocas:5d} -> error (MSE): {costo:.6f}")
    return W, b

In [ ]:
X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]])

# Dos columnas: la primera es AND, la segunda es OR.
# Una sola capa con dos neuronas aprende las dos cosas al tiempo.
Y_dos = np.array([[0, 0],
                  [0, 1],
                  [0, 1],
                  [1, 1]])

print("Entrenando una capa de 2 neuronas (AND y OR simultaneamente):")
W_capa, b_capa = entrenar_capa(X, Y_dos, lr=0.5, epocas=5000)

salidas = sigmoide(X @ W_capa + b_capa)

print("\nRESULTADO - Capa de 2 neuronas")
print("  Entrada     AND esperado / obtenido      OR esperado / obtenido")
for entrada, esperado, obtenido in zip(X, Y_dos, salidas):
    print(f"  {entrada}          {esperado[0]}  /  {obtenido[0]:.3f}"
          f"                 {esperado[1]}  /  {obtenido[1]:.3f}")

print("\nMatriz de pesos W (2 entradas x 2 neuronas):")
print(np.round(W_capa, 3))
print("Sesgos b:", np.round(b_capa, 3))

Entrenando una capa de 2 neuronas (AND y OR simultaneamente):
  Epoca     0 -> error (MSE): 0.206347
  Epoca  1000 -> error (MSE): 0.020040
  Epoca  2000 -> error (MSE): 0.009393
  Epoca  3000 -> error (MSE): 0.005957
  Epoca  4000 -> error (MSE): 0.004312
  Epoca  5000 -> error (MSE): 0.003362

RESULTADO - Capa de 2 neuronas
  Entrada     AND esperado / obtenido      OR esperado / obtenido
  [0 0]          0  /  0.001                 0  /  0.071
  [0 1]          0  /  0.072                 1  /  0.956
  [1 0]          0  /  0.072                 1  /  0.956
  [1 1]          1  /  0.914                 1  /  1.000

Matriz de pesos W (2 entradas x 2 neuronas):
[[4.91  5.644]
 [4.91  5.644]]
Sesgos b: [[-7.461 -2.57 ]]


### 4.2. ¿cómo funciona aqui el XOR?

Cambiar la función de activación no resuelve el problema de fondo. Una sola capa, con escalón o con
sigmoide, sigue trazando una línea recta.

In [ ]:
Y_xor_col = np.array([[0], [1], [1], [0]])   # XOR como columna

print("Entrenando una capa de 1 neurona con XOR:")
W_x, b_x = entrenar_capa(X, Y_xor_col, lr=0.5, epocas=5000)

salidas_xor = sigmoide(X @ W_x + b_x)

print("\nRESULTADO - XOR con una sola capa")
print("  Entrada     Esperado    Obtenido")
for entrada, esperado, obtenido in zip(X, Y_xor_col, salidas_xor):
    print(f"  {entrada}          {esperado[0]}        {obtenido[0]:.4f}")
print("\nTodas las salidas rondan 0.5: la red no logra decidirse. Necesitamos mas capas.")

Entrenando una capa de 1 neurona con XOR:
  Epoca     0 -> error (MSE): 0.274190
  Epoca  1000 -> error (MSE): 0.250000
  Epoca  2000 -> error (MSE): 0.250000
  Epoca  3000 -> error (MSE): 0.250000
  Epoca  4000 -> error (MSE): 0.250000
  Epoca  5000 -> error (MSE): 0.250000

RESULTADO - XOR con una sola capa
  Entrada     Esperado    Obtenido
  [0 0]          0        0.5000
  [0 1]          1        0.5000
  [1 0]          1        0.5000
  [1 1]          0        0.5000

Todas las salidas rondan 0.5: la red no logra decidirse. Necesitamos mas capas.


---
## 5. Red neuronal multicapa

Aquí está la solución. Al agregar una capa oculta entre la entrada y la salida, la red deja de estar
limitada a una línea recta: la primera capa transforma los datos y la segunda ya puede separarlos.

La red que vamos a construir tiene esta topología:

```
   2 entradas  ->  4 neuronas ocultas  ->  1 salida
      X                  Capa 1              Capa 2
```

### ¿Cómo aprende una red multicapa?

En dos movimientos:

1. **Hacia adelante** (*forward*): los datos entran, pasan por la capa oculta, llegan a la salida y se
   obtiene una predicción.
2. **Hacia atrás** (*backpropagation*): se mide el error en la salida y se reparte hacia atrás, capa por
   capa, para saber qué tanta culpa tuvo cada peso. Después cada peso se corrige según su culpa.

Eso es todo. Se repite miles de veces y la red va afinando sus pesos.

In [ ]:
class Capa:
    """Una capa de la red: guarda su matriz de pesos y su vector de sesgos.

    n_entradas : cuantos valores recibe cada neurona
    n_neuronas : cuantas neuronas tiene la capa
    """
    def __init__(self, n_entradas, n_neuronas):
        # Pesos aleatorios entre -1 y 1. Es importante que NO sean todos iguales:
        # si arrancaran todos en cero, todas las neuronas aprenderian exactamente lo mismo.
        self.W = np.random.uniform(-1, 1, size=(n_entradas, n_neuronas))
        self.b = np.zeros((1, n_neuronas))


def crear_red(topologia):
    """Crea la lista de capas a partir de una topologia.

    Ejemplo: topologia = [2, 4, 1] crea una red de 2 entradas, 4 neuronas ocultas y 1 salida.
    """
    return [Capa(topologia[i], topologia[i + 1]) for i in range(len(topologia) - 1)]


# Creamos la red y revisamos que las formas de las matrices sean las correctas
red_ejemplo = crear_red([2, 4, 1])
for i, capa in enumerate(red_ejemplo, start=1):
    print(f"Capa {i}: W con forma {capa.W.shape}, b con forma {capa.b.shape}")

Capa 1: W con forma (2, 4), b con forma (1, 4)
Capa 2: W con forma (4, 1), b con forma (1, 1)


### 5.1. Propagación hacia adelante

Los datos atraviesan la red capa por capa. La salida de una capa se convierte en la entrada de la
siguiente. Guardamos todas las activaciones porque las vamos a necesitar para la retropropagación.

In [ ]:
def adelante(red, X):
    """Pasa los datos por toda la red y devuelve las activaciones de cada capa.

    activaciones[0] son los datos de entrada, y la ultima es la prediccion final.
    """
    activaciones = [X]
    a = X
    for capa in red:
        z = a @ capa.W + capa.b     # suma ponderada
        a = sigmoide(z)             # activacion
        activaciones.append(a)
    return activaciones


# Probamos con la red recien creada (todavia no ha aprendido nada, los pesos son aleatorios)
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
acts = adelante(red_ejemplo, X)

print("Entrada (4 ejemplos x 2 entradas):", acts[0].shape)
print("Salida de la capa oculta         :", acts[1].shape)
print("Salida final                     :", acts[2].shape)
print()
print("Predicciones iniciales (sin entrenar, por eso no significan nada todavia):")
print(np.round(acts[-1].ravel(), 4))

Entrada (4 ejemplos x 2 entradas): (4, 2)
Salida de la capa oculta         : (4, 4)
Salida final                     : (4, 1)

Predicciones iniciales (sin entrenar, por eso no significan nada todavia):
[0.3566 0.3591 0.4138 0.4155]


### 5.2. Retropropagación y entrenamiento

Esta es la parte central del taller. La idea, sin fórmulas complicadas:

- En la **última capa**, el error es directo: lo que predijo menos lo que debía predecir.
- En las **capas anteriores**, no sabemos el error directamente, así que lo estimamos: cada neurona
  oculta tuvo tanta culpa como peso tenía su conexión hacia la neurona que se equivocó.
- Con esa "culpa" (el *delta*) se calcula cuánto mover cada peso y se da un pasito.

In [ ]:
def entrenar_red(red, X, Y, lr=0.5, epocas=20000, mostrar_cada=4000):
    """Entrena la red multicapa con retropropagacion."""
    n = X.shape[0]
    historial = []

    for epoca in range(epocas + 1):

        # ---------- 1) HACIA ADELANTE ----------
        activaciones = adelante(red, X)
        prediccion = activaciones[-1]

        costo = np.mean((prediccion - Y) ** 2)      # error cuadratico medio
        historial.append(costo)

        # ---------- 2) HACIA ATRAS ----------
        deltas = [None] * len(red)

        # Error de la ultima capa
        deltas[-1] = (prediccion - Y) * derivada_sigmoide(prediccion)

        # Se reparte el error hacia las capas anteriores, de la ultima a la primera
        for i in reversed(range(len(red) - 1)):
            deltas[i] = (deltas[i + 1] @ red[i + 1].W.T) * derivada_sigmoide(activaciones[i + 1])

        # ---------- 3) ACTUALIZAR LOS PESOS ----------
        for i, capa in enumerate(red):
            capa.W = capa.W - lr * (activaciones[i].T @ deltas[i]) / n
            capa.b = capa.b - lr * np.mean(deltas[i], axis=0, keepdims=True)

        if epoca % mostrar_cada == 0:
            print(f"  Epoca {epoca:6d} -> error (MSE): {costo:.6f}")

    return historial

### 5.3. Por fin: resolviendo el XOR

El mismo problema que el perceptrón no pudo resolver, ahora con una capa oculta de cuatro neuronas.

In [ ]:
np.random.seed(42)      # para que el resultado sea reproducible

X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]])
Y_xor = np.array([[0], [1], [1], [0]])

# Topologia: 2 entradas -> 4 neuronas ocultas -> 1 salida
red_xor = crear_red([2, 4, 1])

print("Entrenando la red multicapa con XOR:")
historial = entrenar_red(red_xor, X, Y_xor, lr=2.0, epocas=20000, mostrar_cada=4000)

predicciones = adelante(red_xor, X)[-1]

print("\nRESULTADO - XOR con red multicapa")
print("  Entrada     Esperado    Obtenido    Redondeado")
for entrada, esperado, obtenido in zip(X, Y_xor, predicciones):
    print(f"  {entrada}          {esperado[0]}        {obtenido[0]:.4f}        {int(round(obtenido[0]))}")

aciertos = np.sum(np.round(predicciones) == Y_xor)
print(f"\n  Aciertos: {aciertos} de 4")
print(f"  El error bajo de {historial[0]:.4f} a {historial[-1]:.6f}")
print("\n  Lo que el perceptron no pudo resolver, la red multicapa si.")

Entrenando la red multicapa con XOR:
  Epoca      0 -> error (MSE): 0.262916
  Epoca   4000 -> error (MSE): 0.000705
  Epoca   8000 -> error (MSE): 0.000309
  Epoca  12000 -> error (MSE): 0.000195
  Epoca  16000 -> error (MSE): 0.000142
  Epoca  20000 -> error (MSE): 0.000111

RESULTADO - XOR con red multicapa
  Entrada     Esperado    Obtenido    Redondeado
  [0 0]          0        0.0064        0
  [0 1]          1        0.9890        1
  [1 0]          1        0.9897        1
  [1 1]          0        0.0133        0

  Aciertos: 4 de 4
  El error bajo de 0.2629 a 0.000111

  Lo que el perceptron no pudo resolver, la red multicapa si.


### 5.4. ¿Qué aprendió la capa oculta?

Vale la pena mirar qué está haciendo la capa del medio. Cada una de sus cuatro neuronas aprendió a
reaccionar ante una situación distinta, y la capa de salida combina esas cuatro opiniones para dar la
respuesta final. Eso es lo que una sola neurona no podía hacer.

In [ ]:
salida_oculta = adelante(red_xor, X)[1]     # activaciones de la capa oculta

print("Que responde cada neurona oculta ante cada entrada:")
print("  Entrada     N1      N2      N3      N4")
for entrada, fila in zip(X, salida_oculta):
    print(f"  {entrada}    " + "  ".join(f"{v:.3f}" for v in fila))

print("\nPesos de la capa oculta (2 entradas x 4 neuronas):")
print(np.round(red_xor[0].W, 2))
print("\nPesos de la capa de salida (4 neuronas x 1 salida):")
print(np.round(red_xor[1].W, 2))

Que responde cada neurona oculta ante cada entrada:
  Entrada     N1      N2      N3      N4
  [0 0]    0.648  0.106  0.897  0.343
  [0 1]    0.341  0.000  0.057  0.965
  [1 0]    0.535  0.935  1.000  0.964
  [1 1]    0.244  0.021  0.979  0.999

Pesos de la capa oculta (2 entradas x 4 neuronas):
[[-0.47  4.8   6.63  3.95]
 [-1.27 -6.51 -4.97  3.98]]

Pesos de la capa de salida (4 neuronas x 1 salida):
[[ 1.7 ]
 [ 9.57]
 [-9.76]
 [ 4.64]]


---
## 6. Caso aplicado: predecir si un estudiante aprueba

Las compuertas lógicas sirven para entender el mecanismo, pero veamos la misma red con un problema más
cercano a la realidad.

**El problema:** a partir de las horas de estudio semanales y el porcentaje de asistencia,
predecir si un estudiante aprueba la materia (1) o no (0).

**Un detalle importante:** las horas van de 0 a 10 y la asistencia de 0 a 100. Si se dejan así, la
asistencia pesaría diez veces más solo por tener números más grandes. Por eso se normalizan ambas
columnas a una escala de 0 a 1 antes de entrenar.

In [ ]:
np.random.seed(42)

# Datos predefinidos: [horas de estudio, % de asistencia]
datos = np.array([
    [1.0, 40],
    [2.0, 55],
    [1.5, 70],
    [3.0, 50],
    [3.5, 65],
    [2.0, 90],
    [4.0, 60],
    [4.5, 75],
    [5.0, 70],
    [6.0, 80],
    [7.0, 90],
    [8.0, 95],
])

# 1 = aprueba, 0 = no aprueba
etiquetas = np.array([[0], [0], [0], [0], [0], [0],
                      [1], [1], [1], [1], [1], [1]])

# Normalizacion: llevamos todo a una escala de 0 a 1
maximos = np.array([10.0, 100.0])
X_est = datos / maximos

print("Datos originales y normalizados:")
print("  Horas  Asistencia  ->  Normalizado      Aprueba")
for original, normalizado, etiqueta in zip(datos, X_est, etiquetas):
    print(f"  {original[0]:4.1f}     {original[1]:5.1f}     ->  "
          f"[{normalizado[0]:.2f}, {normalizado[1]:.2f}]         {etiqueta[0]}")

Datos originales y normalizados:
  Horas  Asistencia  ->  Normalizado      Aprueba
   1.0      40.0     ->  [0.10, 0.40]         0
   2.0      55.0     ->  [0.20, 0.55]         0
   1.5      70.0     ->  [0.15, 0.70]         0
   3.0      50.0     ->  [0.30, 0.50]         0
   3.5      65.0     ->  [0.35, 0.65]         0
   2.0      90.0     ->  [0.20, 0.90]         0
   4.0      60.0     ->  [0.40, 0.60]         1
   4.5      75.0     ->  [0.45, 0.75]         1
   5.0      70.0     ->  [0.50, 0.70]         1
   6.0      80.0     ->  [0.60, 0.80]         1
   7.0      90.0     ->  [0.70, 0.90]         1
   8.0      95.0     ->  [0.80, 0.95]         1


In [ ]:
# La misma red de antes, con la misma topologia
red_est = crear_red([2, 4, 1])

print("Entrenando la red con los datos de los estudiantes:")
hist_est = entrenar_red(red_est, X_est, etiquetas, lr=2.0, epocas=20000, mostrar_cada=4000)

pred_est = adelante(red_est, X_est)[-1]
clasificacion = np.round(pred_est)

print("\nRESULTADO - Estudiantes")
print("  Horas  Asistencia   Real   Probabilidad   Prediccion")
for original, real, prob, pred in zip(datos, etiquetas, pred_est, clasificacion):
    marca = "OK" if real[0] == pred[0] else "FALLA"
    print(f"  {original[0]:4.1f}     {original[1]:5.1f}       {real[0]}       {prob[0]:.4f}"
          f"          {int(pred[0])}   {marca}")

exactitud = np.mean(clasificacion == etiquetas) * 100
print(f"\n  Exactitud sobre los datos de entrenamiento: {exactitud:.1f} %")

Entrenando la red con los datos de los estudiantes:
  Epoca      0 -> error (MSE): 0.258958
  Epoca   4000 -> error (MSE): 0.006361
  Epoca   8000 -> error (MSE): 0.001875
  Epoca  12000 -> error (MSE): 0.000948
  Epoca  16000 -> error (MSE): 0.000601
  Epoca  20000 -> error (MSE): 0.000429

RESULTADO - Estudiantes
  Horas  Asistencia   Real   Probabilidad   Prediccion
   1.0      40.0       0       0.0000          0   OK
   2.0      55.0       0       0.0000          0   OK
   1.5      70.0       0       0.0000          0   OK
   3.0      50.0       0       0.0143          0   OK
   3.5      65.0       0       0.0489          0   OK
   2.0      90.0       0       0.0000          0   OK
   4.0      60.0       1       0.9514          1   OK
   4.5      75.0       1       0.9860          1   OK
   5.0      70.0       1       0.9999          1   OK
   6.0      80.0       1       1.0000          1   OK
   7.0      90.0       1       1.0000          1   OK
   8.0      95.0       1       1.0

### 6.1. Probando con estudiantes nuevos

Lo interesante de un modelo entrenado es usarlo con casos que nunca vio. Estos tres estudiantes no
estaban en los datos de entrenamiento.

In [ ]:
nuevos = np.array([
    [5.0, 85],    # estudia bastante y asiste mucho
    [1.0, 45],    # estudia poco y falta mucho
    [3.8, 62],    # un caso en la frontera
])

X_nuevos = nuevos / maximos                 # se normaliza igual que en el entrenamiento
pred_nuevos = adelante(red_est, X_nuevos)[-1]

print("Predicciones para estudiantes nuevos:")
print("  Horas  Asistencia   Probabilidad de aprobar   Prediccion")
for original, prob in zip(nuevos, pred_nuevos):
    resultado = "APRUEBA" if prob[0] >= 0.5 else "NO APRUEBA"
    print(f"  {original[0]:4.1f}     {original[1]:5.1f}            {prob[0]*100:6.2f} %"
          f"              {resultado}")

print("\n  Nota: el tercer caso queda cerca del 50 %, que es justo la frontera de decision.")
print("  La red no solo responde si o no: tambien dice que tan segura esta.")

Predicciones para estudiantes nuevos:
  Horas  Asistencia   Probabilidad de aprobar   Prediccion
   5.0      85.0             99.86 %              APRUEBA
   1.0      45.0              0.00 %              NO APRUEBA
   3.8      62.0             64.43 %              APRUEBA

  Nota: el tercer caso queda cerca del 50 %, que es justo la frontera de decision.
  La red no solo responde si o no: tambien dice que tan segura esta.


---
## 7. Comparación de los tres modelos

| | **Perceptrón** | **Red de una capa** | **Red multicapa** |
|---|---|---|---|
| **Neuronas** | 1 | Varias, en paralelo | Varias, en capas sucesivas |
| **Activación** | Escalón (0 o 1) | Sigmoide (entre 0 y 1) | Sigmoide (entre 0 y 1) |
| **Cómo aprende** | Regla del perceptrón | Descenso del gradiente | Retropropagación |
| **Pesos** | Un vector | Una matriz | Una matriz por capa |
| **Resuelve AND / OR** | Sí | Sí | Sí |
| **Resuelve XOR** | **No** | **No** | **Sí** |
| **Qué puede separar** | Solo con una línea recta | Solo con una línea recta | Formas más complejas |

## 8. Conclusiones

- **Una red neuronal no es más que multiplicaciones y sumas de matrices.** Al programarla desde cero queda
  claro que detrás de todo el asunto hay álgebra lineal: pesos que multiplican entradas, sesgos que se
  suman y una función que decide la salida. No hay ninguna magia.

- **El perceptrón sirve, pero hasta cierto punto.** Resolvió AND y OR sin problema, pero fracasó con XOR
  porque solo puede trazar una línea recta. Esa limitación, que se comprobó en el punto 3.4, es
  exactamente la razón histórica por la que se inventaron las redes multicapa.

- **La capa oculta es lo que cambia todo.** Con solo agregar cuatro neuronas intermedias, la misma red
  pasó de no poder resolver XOR a resolverlo con el 100 % de aciertos. Ahí está, en pequeño, la idea
  central del Deep Learning: apilar capas permite aprender relaciones que una sola capa no alcanza.

- **La vectorización no es un lujo.** En la comparación del punto 2.5, NumPy resolvió la misma operación
  cientos de veces más rápido que un ciclo de Python. En una red real, con miles de datos y miles de
  épocas, esa diferencia decide si el entrenamiento toma minutos o días.

- **Normalizar los datos importa.** En el caso de los estudiantes, dejar la asistencia en escala de 0 a 100
  y las horas de 0 a 10 habría hecho que la red le diera más peso a la asistencia solo por tener números
  más grandes. Normalizar puso ambas variables en igualdad de condiciones.

- **La red entrega probabilidades, no solo respuestas.** Gracias a la sigmoide, el modelo no dice
  únicamente "aprueba" o "no aprueba", sino qué tan seguro está. El estudiante que quedó cerca del 50 % es
  un caso dudoso, y eso es información útil que una respuesta de sí o no habría ocultado.